In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

In [2]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "datasets",
    "processed",
    "stroke"
)

MODEL_DIR = os.path.join(
    BASE_DIR,
    "models",
    "stroke"
)

os.makedirs(MODEL_DIR, exist_ok=True)

In [3]:
X_train = pd.read_csv(
    os.path.join(PROCESSED_DIR, "x_train.csv")
)

X_test = pd.read_csv(
    os.path.join(PROCESSED_DIR, "x_test.csv")
)

y_train = pd.read_csv(
    os.path.join(PROCESSED_DIR, "y_train.csv")
).squeeze()

y_test = pd.read_csv(
    os.path.join(PROCESSED_DIR, "y_test.csv")
).squeeze()

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training data: (4088, 21)
Testing data: (1022, 21)

Training target distribution:
stroke
0    3889
1     199
Name: count, dtype: int64

Testing target distribution:
stroke
0    972
1     50
Name: count, dtype: int64


In [4]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Non-stroke cases:", negative_count)
print("Stroke cases:", positive_count)
print("Scale positive weight:", scale_pos_weight)

Non-stroke cases: 3889
Stroke cases: 199
Scale positive weight: 19.542713567839197


In [5]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "SVM": SVC(
        probability=True,
        class_weight="balanced",
        random_state=42
    ),

    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42
    )
}

print("Models created successfully!")

Models created successfully!


In [6]:
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    
    model.fit(X_train, y_train)
    
    trained_models[name] = model

print("\nAll models trained successfully!")

Training Logistic Regression...
Training Decision Tree...
Training Random Forest...
Training SVM...


d:\Explainable-AI-CDSS\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Training XGBoost...

All models trained successfully!


In [7]:
model_paths = {
    "Logistic Regression": "logistic_regression.pkl",
    "Decision Tree": "decision_tree.pkl",
    "Random Forest": "random_forest.pkl",
    "SVM": "svm.pkl",
    "XGBoost": "xgboost.pkl"
}

for name, filename in model_paths.items():
    path = os.path.join(MODEL_DIR, filename)
    joblib.dump(trained_models[name], path)
    print(f"Saved: {filename}")

Saved: logistic_regression.pkl
Saved: decision_tree.pkl
Saved: random_forest.pkl
Saved: svm.pkl
Saved: xgboost.pkl


In [8]:
for name, model in trained_models.items():
    predictions = model.predict(X_test)
    
    print(f"\n{name}")
    print("Predicted classes:")
    print(pd.Series(predictions).value_counts())


Logistic Regression
Predicted classes:
0    733
1    289
Name: count, dtype: int64

Decision Tree
Predicted classes:
0    990
1     32
Name: count, dtype: int64

Random Forest
Predicted classes:
0    994
1     28
Name: count, dtype: int64

SVM
Predicted classes:
0    779
1    243
Name: count, dtype: int64

XGBoost
Predicted classes:
0    840
1    182
Name: count, dtype: int64


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [10]:
results = []

for name, model in trained_models.items():
    
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_prob)
    
    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

results_df = pd.DataFrame(results)

results_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.746575,0.138408,0.80,0.235988,0.843827
1,Decision Tree,0.929550,0.156250,0.10,0.121951,0.536111
2,Random Forest,0.935421,0.214286,0.12,0.153846,0.775453
3,SVM,0.772016,0.123457,0.60,0.204778,0.789239
4,XGBoost,0.839530,0.186813,0.68,0.293103,0.842490


In [11]:
metrics_display = results_df.copy()

metric_columns = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC-AUC"
]

metrics_display[metric_columns] = (
    metrics_display[metric_columns] * 100
).round(2)

metrics_display

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,74.66,13.84,80.0,23.60,84.38
1,Decision Tree,92.95,15.62,10.0,12.20,53.61
2,Random Forest,93.54,21.43,12.0,15.38,77.55
3,SVM,77.20,12.35,60.0,20.48,78.92
4,XGBoost,83.95,18.68,68.0,29.31,84.25


In [12]:
print("=" * 70)
print("STROKE MODEL COMPARISON")
print("=" * 70)

for _, row in results_df.iterrows():
    print(f"\n{row['Model']}")
    print(f"Accuracy : {row['Accuracy'] * 100:.2f}%")
    print(f"Precision: {row['Precision'] * 100:.2f}%")
    print(f"Recall   : {row['Recall'] * 100:.2f}%")
    print(f"F1 Score : {row['F1 Score'] * 100:.2f}%")
    print(f"ROC-AUC  : {row['ROC-AUC'] * 100:.2f}%")

STROKE MODEL COMPARISON

Logistic Regression
Accuracy : 74.66%
Precision: 13.84%
Recall   : 80.00%
F1 Score : 23.60%
ROC-AUC  : 84.38%

Decision Tree
Accuracy : 92.95%
Precision: 15.62%
Recall   : 10.00%
F1 Score : 12.20%
ROC-AUC  : 53.61%

Random Forest
Accuracy : 93.54%
Precision: 21.43%
Recall   : 12.00%
F1 Score : 15.38%
ROC-AUC  : 77.55%

SVM
Accuracy : 77.20%
Precision: 12.35%
Recall   : 60.00%
F1 Score : 20.48%
ROC-AUC  : 78.92%

XGBoost
Accuracy : 83.95%
Precision: 18.68%
Recall   : 68.00%
F1 Score : 29.31%
ROC-AUC  : 84.25%
